# Surf Rating Classification — Model Experiments

Baseline comparison of five classifiers for predicting surf quality ratings.  
Training data: `data/processed/training_set_enriched.csv` (303,375 rows, 7 classes).

In [1]:
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV, train_test_split

RANDOM_STATE = 42

## 1. Data Loading and Cleaning

In [2]:
import pathlib

# Notebooks don't have __file__; walk up from the kernel's cwd until
# we find the directory that contains data/processed/
_here = pathlib.Path().resolve()
_root = _here
for _ in range(6):
    if (_root / 'data' / 'processed').exists():
        break
    _root = _root.parent
else:
    raise FileNotFoundError(
        f'Could not locate project root (data/processed/) starting from {_here}'
    )

PROJECT_ROOT = _root
DATA_PATH    = PROJECT_ROOT / 'data' / 'processed' / 'training_set_enriched.csv'
print(f'Project root: {PROJECT_ROOT}')
print(f'Data path:    {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

Project root: /Users/yraclimaco/Desktop/Surf-Cast-SD
Data path:    /Users/yraclimaco/Desktop/Surf-Cast-SD/data/processed/training_set_enriched.csv
Loaded: 303,375 rows x 20 columns


,timestamp_utc,station,WVHT,DPD,MWD,APD,tide_height_m,wind_speed_mph,wind_direction_deg,break_id,rating_key,rating_value,sunrise_hour_utc,sunset_hour_utc,day_length_hours,hours_since_sunrise,season_sin,season_cos,temp_c,humidity_pct
0,2022-01-01 08:26:00+00:00,46232,2.18,9.09,275.0,6.79,0.7449,3.35541,210.0,blacks,POOR,1.0,14.85,0.883333,10.039461,0.0,0.017213,0.999852,16.1,62.350548
1,2022-01-01 08:26:00+00:00,46254,1.35,9.09,282.0,6.26,0.7449,3.35541,210.0,blacks,POOR,1.0,14.85,0.883333,10.039461,0.0,0.017213,0.999852,16.1,62.350548
2,2022-01-01 08:26:00+00:00,46232,2.18,9.09,275.0,6.79,0.7449,3.35541,210.0,la_jolla_shores,POOR_TO_FAIR,2.0,14.85,0.883333,10.039461,0.0,0.017213,0.999852,16.1,62.350548


In [3]:
# is_calm flag before filling NaNs so it reflects the raw zero
df['is_calm'] = (df['wind_speed_mph'] == 0).astype(int)

# Fill wind_direction_deg NaNs with 0 (calm / no direction)
df['wind_direction_deg'] = df['wind_direction_deg'].fillna(0)

# Fill remaining numeric NaNs with column median
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    n_null = df[col].isna().sum()
    if n_null:
        df[col] = df[col].fillna(df[col].median())
        print(f'  filled {n_null:,} NaNs in "{col}" with median')

assert len(df) == 303_375, f'Expected 303,375 rows, got {len(df):,}'
print(f'\nRows after cleaning: {len(df):,}  OK')

  filled 69 NaNs in "WVHT" with median
  filled 132 NaNs in "DPD" with median
  filled 132 NaNs in "MWD" with median
  filled 69 NaNs in "APD" with median
  filled 372 NaNs in "wind_speed_mph" with median
  filled 360 NaNs in "temp_c" with median
  filled 420 NaNs in "humidity_pct" with median

Rows after cleaning: 303,375  OK


In [4]:
CLASS_ORDER = ['EPIC', 'GOOD', 'FAIR_TO_GOOD', 'FAIR', 'POOR_TO_FAIR', 'POOR', 'VERY_POOR']

dist = df['rating_key'].value_counts().reindex(CLASS_ORDER)
pct  = (dist / len(df) * 100).round(2)

print('Class distribution:')
print(pd.DataFrame({'count': dist, 'pct': pct}).to_string())

Class distribution:
               count    pct
rating_key                 
EPIC              24   0.01
GOOD             668   0.22
FAIR_TO_GOOD   13880   4.58
FAIR          142579  47.00
POOR_TO_FAIR  121207  39.95
POOR           22801   7.52
VERY_POOR       2216   0.73


## 2. Feature Preparation

In [5]:
FEATURES = [
    'WVHT', 'DPD', 'MWD', 'APD', 'tide_height_m',
    'wind_speed_mph', 'wind_direction_deg', 'is_calm',
    'sunrise_hour_utc', 'sunset_hour_utc', 'day_length_hours',
    'hours_since_sunrise', 'season_sin', 'season_cos',
    'temp_c', 'humidity_pct',
]
TARGET = 'rating_key'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)
print(f'Train: {X_train.shape}   Test: {X_test.shape}')

Train: (212362, 16)   Test: (91013, 16)


## 3. Model Training with Hyperparameter Tuning

Each model uses `RandomizedSearchCV` with `cv=3`, `n_iter=20`, `scoring='f1_weighted'`.

In [6]:
results = []
timing  = {}

SEARCH_KWARGS = dict(cv=3, n_iter=20, scoring='f1_weighted',
                     random_state=RANDOM_STATE, n_jobs=-1, verbose=1)

def evaluate(name, best_est, best_params, elapsed, X_te=None, y_te=None):
    if X_te is None: X_te = X_test
    if y_te is None: y_te = y_test
    y_pred = best_est.predict(X_te)
    acc    = accuracy_score(y_te, y_pred)
    f1_w   = f1_score(y_te, y_pred, average='weighted', labels=CLASS_ORDER, zero_division=0)
    f1_m   = f1_score(y_te, y_pred, average='macro',    labels=CLASS_ORDER, zero_division=0)
    f1_pc  = f1_score(y_te, y_pred, average=None,       labels=CLASS_ORDER, zero_division=0)

    row = {
        'model':       name,
        'best_params': str(best_params),
        'accuracy':    round(acc, 4),
        'f1_weighted': round(f1_w, 4),
        'f1_macro':    round(f1_m, 4),
        'fit_time_s':  round(elapsed, 1),
    }
    for cls, score in zip(CLASS_ORDER, f1_pc):
        row[f'f1_{cls}'] = round(score, 4)

    results.append(row)
    timing[name] = elapsed

    print(f'\n[{name}]  time={elapsed:.1f}s')
    print(f'  accuracy={acc:.4f}  f1_weighted={f1_w:.4f}  f1_macro={f1_m:.4f}')
    print(f'  best_params: {best_params}')
    print(classification_report(y_te, y_pred, labels=CLASS_ORDER,
                                target_names=CLASS_ORDER, zero_division=0))

In [7]:
# Random Forest
rf_param_dist = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2'],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    rf_param_dist, **SEARCH_KWARGS,
)

t0 = time.time()
rf_search.fit(X_train, y_train)
rf_elapsed = time.time() - t0

rf_best = rf_search.best_estimator_
evaluate('RandomForest', rf_best, rf_search.best_params_, rf_elapsed)

Fitting 3 folds for each of 20 candidates, totalling 60 fits

[RandomForest]  time=319.0s
  accuracy=0.6524  f1_weighted=0.6631  f1_macro=0.5412
  best_params: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 20}
              precision    recall  f1-score   support

        EPIC       0.32      1.00      0.48         7
        GOOD       0.29      0.84      0.43       201
FAIR_TO_GOOD       0.33      0.68      0.45      4164
        FAIR       0.77      0.65      0.71     42774
POOR_TO_FAIR       0.69      0.65      0.67     36362
        POOR       0.41      0.61      0.49      6840
   VERY_POOR       0.44      0.78      0.56       665

    accuracy                           0.65     91013
   macro avg       0.46      0.75      0.54     91013
weighted avg       0.69      0.65      0.66     91013



In [8]:
# HistGradientBoosting
hgb_param_dist = {
    'max_iter':          [100, 200, 300],
    'max_depth':         [None, 5, 10, 15],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'min_samples_leaf':  [10, 20, 50],
    'l2_regularization': [0, 0.1, 1.0],
}

hgb_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    hgb_param_dist, **SEARCH_KWARGS,
)

t0 = time.time()
hgb_search.fit(X_train, y_train)
hgb_elapsed = time.time() - t0

hgb_best = hgb_search.best_estimator_
evaluate('HistGradientBoosting', hgb_best, hgb_search.best_params_, hgb_elapsed)

Fitting 3 folds for each of 20 candidates, totalling 60 fits

[HistGradientBoosting]  time=65.9s
  accuracy=0.5508  f1_weighted=0.5729  f1_macro=0.4714
  best_params: {'min_samples_leaf': 10, 'max_iter': 100, 'max_depth': 10, 'learning_rate': 0.2, 'l2_regularization': 1.0}
              precision    recall  f1-score   support

        EPIC       0.29      0.86      0.43         7
        GOOD       0.24      0.96      0.38       201
FAIR_TO_GOOD       0.23      0.82      0.36      4164
        FAIR       0.73      0.51      0.60     42774
POOR_TO_FAIR       0.66      0.54      0.59     36362
        POOR       0.32      0.66      0.43      6840
   VERY_POOR       0.35      0.91      0.51       665

    accuracy                           0.55     91013
   macro avg       0.40      0.75      0.47     91013
weighted avg       0.65      0.55      0.57     91013



In [9]:
# GradientBoosting — 50k row subsample from X_train for speed
# GradientBoostingClassifier has no class_weight parameter
gb_sample_idx = (
    pd.Series(range(len(X_train)))
      .sample(n=50_000, random_state=RANDOM_STATE)
      .values
)
X_train_gb = X_train.iloc[gb_sample_idx]
y_train_gb = y_train.iloc[gb_sample_idx]
print(f'GradientBoosting subsample: {X_train_gb.shape}')

gb_param_dist = {
    'n_estimators':      [100, 200],
    'max_depth':         [3, 5, 7],
    'learning_rate':     [0.05, 0.1, 0.2],
    'min_samples_split': [2, 5],
    'subsample':         [0.8, 1.0],
}

gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    gb_param_dist, **SEARCH_KWARGS,
)

t0 = time.time()
gb_search.fit(X_train_gb, y_train_gb)
gb_elapsed = time.time() - t0

gb_best = gb_search.best_estimator_
evaluate('GradientBoosting', gb_best, gb_search.best_params_, gb_elapsed)

GradientBoosting subsample: (50000, 16)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

[GradientBoosting]  time=2142.2s
  accuracy=0.6813  f1_weighted=0.6696  f1_macro=0.3790
  best_params: {'subsample': 0.8, 'n_estimators': 200, 'min_samples_split': 2, 'max_depth': 7, 'learning_rate': 0.05}
              precision    recall  f1-score   support

        EPIC       0.00      0.00      0.00         7
        GOOD       0.12      0.15      0.13       201
FAIR_TO_GOOD       0.43      0.19      0.26      4164
        FAIR       0.71      0.79      0.75     42774
POOR_TO_FAIR       0.67      0.69      0.68     36362
        POOR       0.56      0.31      0.40      6840
   VERY_POOR       0.49      0.38      0.43       665

    accuracy                           0.68     91013
   macro avg       0.43      0.36      0.38     91013
weighted avg       0.67      0.68      0.67     91013



In [10]:
# Logistic Regression
lr_param_dist = {
    'C':           [0.01, 0.1, 1, 10, 100],
    'penalty':     ['l1', 'l2'],
    'multi_class': ['multinomial'],
}

lr_search = RandomizedSearchCV(
    LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='saga',
        random_state=RANDOM_STATE,
    ),
    lr_param_dist, **SEARCH_KWARGS,
)

t0 = time.time()
lr_search.fit(X_train, y_train)
lr_elapsed = time.time() - t0

lr_best = lr_search.best_estimator_
evaluate('LogisticRegression', lr_best, lr_search.best_params_, lr_elapsed)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


/Users/yraclimaco/Desktop/Surf-Cast-SD/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/yraclimaco/Desktop/Surf-Cast-SD/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/yraclimaco/Desktop/Surf-Cast-SD/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/yraclimaco/Desktop/Surf-Cast-SD/.venv/l


[LogisticRegression]  time=502.4s
  accuracy=0.3470  f1_weighted=0.3871  f1_macro=0.1969
  best_params: {'penalty': 'l1', 'multi_class': 'multinomial', 'C': 10}
              precision    recall  f1-score   support

        EPIC       0.00      1.00      0.00         7
        GOOD       0.01      0.32      0.03       201
FAIR_TO_GOOD       0.08      0.40      0.14      4164
        FAIR       0.59      0.27      0.37     42774
POOR_TO_FAIR       0.54      0.39      0.45     36362
        POOR       0.24      0.60      0.34      6840
   VERY_POOR       0.94      0.02      0.04       665

    accuracy                           0.35     91013
   macro avg       0.34      0.43      0.20     91013
weighted avg       0.52      0.35      0.39     91013



In [11]:
# Gaussian Naive Bayes — no class_weight support
gnb_param_dist = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5],
}

gnb_search = RandomizedSearchCV(
    GaussianNB(),
    gnb_param_dist, **SEARCH_KWARGS,
)

t0 = time.time()
gnb_search.fit(X_train, y_train)
gnb_elapsed = time.time() - t0

gnb_best = gnb_search.best_estimator_
evaluate('GaussianNB', gnb_best, gnb_search.best_params_, gnb_elapsed)

Fitting 3 folds for each of 5 candidates, totalling 15 fits

[GaussianNB]  time=1.5s
  accuracy=0.4723  f1_weighted=0.4740  f1_macro=0.2641
  best_params: {'var_smoothing': 1e-05}
              precision    recall  f1-score   support

        EPIC       0.04      1.00      0.08         7
        GOOD       0.03      0.47      0.06       201
FAIR_TO_GOOD       0.16      0.55      0.25      4164
        FAIR       0.61      0.36      0.45     42774
POOR_TO_FAIR       0.54      0.67      0.60     36362
        POOR       0.32      0.07      0.12      6840
   VERY_POOR       0.18      0.66      0.28       665

    accuracy                           0.47     91013
   macro avg       0.27      0.54      0.26     91013
weighted avg       0.54      0.47      0.47     91013



## 4. Results Summary

In [12]:
results_df = pd.DataFrame(results)

display_cols = (
    ['model', 'accuracy', 'f1_weighted', 'f1_macro', 'fit_time_s']
    + [f'f1_{c}' for c in CLASS_ORDER]
)

print('=== Model Comparison ===')
print(results_df[display_cols].to_string(index=False))

results_df[display_cols]

=== Model Comparison ===
               model  accuracy  f1_weighted  f1_macro  fit_time_s  f1_EPIC  f1_GOOD  f1_FAIR_TO_GOOD  f1_FAIR  f1_POOR_TO_FAIR  f1_POOR  f1_VERY_POOR
        RandomForest    0.6524       0.6631    0.5412       319.0   0.4828   0.4273           0.4473   0.7078           0.6708   0.4918        0.5607
HistGradientBoosting    0.5508       0.5729    0.4714        65.9   0.4286   0.3784           0.3576   0.6023           0.5918   0.4319        0.5090
    GradientBoosting    0.6813       0.6696    0.3790      2142.2   0.0000   0.1316           0.2624   0.7466           0.6837   0.4012        0.4275
  LogisticRegression    0.3470       0.3871    0.1969       502.4   0.0042   0.0260           0.1402   0.3680           0.4548   0.3410        0.0441
          GaussianNB    0.4723       0.4740    0.2641         1.5   0.0833   0.0632           0.2484   0.4499           0.6007   0.1196        0.2833


,model,accuracy,f1_weighted,f1_macro,fit_time_s,f1_EPIC,f1_GOOD,f1_FAIR_TO_GOOD,f1_FAIR,f1_POOR_TO_FAIR,f1_POOR,f1_VERY_POOR
0,RandomForest,0.6524,0.6631,0.5412,319.0,0.4828,0.4273,0.4473,0.7078,0.6708,0.4918,0.5607
1,HistGradientBoosting,0.5508,0.5729,0.4714,65.9,0.4286,0.3784,0.3576,0.6023,0.5918,0.4319,0.5090
2,GradientBoosting,0.6813,0.6696,0.3790,2142.2,0.0000,0.1316,0.2624,0.7466,0.6837,0.4012,0.4275
3,LogisticRegression,0.3470,0.3871,0.1969,502.4,0.0042,0.0260,0.1402,0.3680,0.4548,0.3410,0.0441
4,GaussianNB,0.4723,0.4740,0.2641,1.5,0.0833,0.0632,0.2484,0.4499,0.6007,0.1196,0.2833


In [13]:
print('=== Feature Importances ===')

model_map = {
    'RandomForest':         rf_best,
    'HistGradientBoosting': hgb_best,
    'GradientBoosting':     gb_best,
}

for name, est in model_map.items():
    if not hasattr(est, 'feature_importances_'):
        print(f'\n{name}: feature_importances_ not available')
        continue
    imp = pd.Series(est.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print(f'\n{name}:')
    print(imp.to_string())

=== Feature Importances ===

RandomForest:
APD                    0.146062
humidity_pct           0.099505
wind_speed_mph         0.088917
season_cos             0.082856
day_length_hours       0.078253
sunrise_hour_utc       0.072781
WVHT                   0.071281
temp_c                 0.065388
tide_height_m          0.060684
season_sin             0.052177
sunset_hour_utc        0.050229
wind_direction_deg     0.035728
hours_since_sunrise    0.034210
DPD                    0.030123
MWD                    0.029048
is_calm                0.002759

HistGradientBoosting: feature_importances_ not available

GradientBoosting:
APD                    0.173036
wind_speed_mph         0.126990
WVHT                   0.098597
day_length_hours       0.077271
humidity_pct           0.071132
MWD                    0.065795
tide_height_m          0.065633
wind_direction_deg     0.059598
hours_since_sunrise    0.055124
temp_c                 0.053052
season_sin             0.044284
season_cos      

## 5. APD vs DPD Importance Comparison

In [14]:
print('=== APD vs DPD Feature Importance ===')
apd_idx = FEATURES.index('APD')
dpd_idx = FEATURES.index('DPD')

apd_dpd_rows = []

for name, est in model_map.items():
    if not hasattr(est, 'feature_importances_'):
        print(f'{name}: feature_importances_ not available — skipping')
        continue
    imp   = est.feature_importances_
    apd   = imp[apd_idx]
    dpd   = imp[dpd_idx]
    ratio = apd / dpd if dpd > 0 else float('inf')
    apd_dpd_rows.append({'model': name, 'APD': round(apd, 6),
                          'DPD': round(dpd, 6), 'APD/DPD': round(ratio, 3)})
    print(f'{name:25s}  APD={apd:.6f}  DPD={dpd:.6f}  APD/DPD={ratio:.3f}')

apd_dpd_df = pd.DataFrame(apd_dpd_rows)
apd_dpd_df

=== APD vs DPD Feature Importance ===
RandomForest               APD=0.146062  DPD=0.030123  APD/DPD=4.849
HistGradientBoosting: feature_importances_ not available — skipping
GradientBoosting           APD=0.173036  DPD=0.027697  APD/DPD=6.247


,model,APD,DPD,APD/DPD
0,RandomForest,0.146062,0.030123,4.849
1,GradientBoosting,0.173036,0.027697,6.247


## 6. Conclusions

**Winner: RandomForest** on macro F1, the honest metric that weights all 7 classes equally. GradientBoosting looks best on accuracy but never predicts EPIC or GOOD so macro F1 drops to 0.379.

**Key issue:** EPIC has only 20 training examples out of 303k rows. Recommend collapsing GOOD+EPIC before the next round of models.
### Best model — Weighted F1
Tree-based ensemble models (RandomForest / HistGradientBoosting) typically dominate on structured tabular data. The winner on weighted F1 reflects performance weighted by class frequency, so the majority class (`POOR_TO_FAIR` at ~47 %) has the strongest influence on this score.

### Best model — Macro F1
Macro F1 penalises rare classes equally. Models trained with `class_weight='balanced'` (RF, HGB, LR) tend to score higher here because they are pushed to attend to minority classes like `EPIC` and `VERY_POOR`.

### APD vs DPD across models
APD (Average Period) aggregates all wave energy while DPD (Dominant Period) singles out the primary swell. If APD/DPD > 1 consistently across tree models, APD is the more globally predictive period feature — potentially because mixed-swell conditions matter more for surf quality than peak swell alone.

### Class imbalance
With ~47 % of labels in `POOR_TO_FAIR`, raw accuracy is misleading. Macro F1 is the better headline metric. Per-class F1 for `EPIC` and `VERY_POOR` reveals whether a model is truly multi-class or collapsing to a majority-class predictor. `class_weight='balanced'` significantly helps minority-class recall at the cost of some majority-class precision.

In [15]:
results_dir = PROJECT_ROOT / 'models' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
out_path = results_dir / 'baseline_results.csv'
results_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

print('\n=== Fit times ===')
for name, t in timing.items():
    print(f'  {name:25s}: {t:.1f}s')

print('\n=== Top model by weighted F1 ===')
best_w = results_df.loc[results_df['f1_weighted'].idxmax(), ['model', 'f1_weighted']]
print(f'  {best_w["model"]}  ({best_w["f1_weighted"]:.4f})')

print('\n=== Top model by macro F1 ===')
best_m = results_df.loc[results_df['f1_macro'].idxmax(), ['model', 'f1_macro']]
print(f'  {best_m["model"]}  ({best_m["f1_macro"]:.4f})')

Saved: /Users/yraclimaco/Desktop/Surf-Cast-SD/models/results/baseline_results.csv

=== Fit times ===
  RandomForest             : 319.0s
  HistGradientBoosting     : 65.9s
  GradientBoosting         : 2142.2s
  LogisticRegression       : 502.4s
  GaussianNB               : 1.5s

=== Top model by weighted F1 ===
  GradientBoosting  (0.6696)

=== Top model by macro F1 ===
  RandomForest  (0.5412)


In [16]:
import joblib
import os

os.makedirs('models/artifacts', exist_ok=True)
joblib.dump(rf_best, 'models/artifacts/surf_model.pkl')
print('Saved:', round(os.path.getsize('models/artifacts/surf_model.pkl') / 1e6, 1), 'MB')

Saved: 286.1 MB
